# Traffic Sign Recognition - Strong Model Training
EfficientNet-B0 + Heavy Augmentation | GPU Recommended

In [ ]:
!pip install -q torch torchvision opencv-python-headless tqdm

## 1. Setup & Config

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
import cv2
import numpy as np
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

BATCH_SIZE = 32
EPOCHS = 12
IMAGE_SIZE = 224
LR = 0.001
DATASET_PATH = '/content/Dataset'  # Upload your Dataset folder here

## 2. Dataset Class

In [ ]:
class TrafficSignDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.data = []
        classes = sorted(os.listdir(root_dir), key=lambda x: int(x))
        for label, cls in enumerate(classes):
            for img in os.listdir(os.path.join(root_dir, cls)):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.data.append((os.path.join(root_dir, cls, img), label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        return img, label

## 3. Transforms (Heavy Augmentation)

In [ ]:
train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(25),
    transforms.RandomAffine(0, translate=(0.15, 0.15), scale=(0.75, 1.25)),
    transforms.ColorJitter(0.5, 0.5, 0.5, 0.15),
    transforms.RandomHorizontalFlip(0.1),
    transforms.GaussianBlur(3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## 4. Load Data

In [ ]:
full = TrafficSignDataset(DATASET_PATH)
train_size = int(0.85 * len(full))
train_ds, val_ds = random_split(full, [train_size, len(full) - train_size])
train_ds.dataset.transform = train_tf
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

## 5. Model - EfficientNet-B0

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 43)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 6. Training

In [ ]:
def train_one_epoch():
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc='Train'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum / len(train_loader), correct / total

def validate():
    model.eval()
    loss_sum, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc='Val'):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            loss = criterion(out, labels)
            loss_sum += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / len(val_loader), correct / total

best_acc = 0
for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    tr_loss, tr_acc = train_one_epoch()
    val_loss, val_acc = validate()
    scheduler.step()
    print(f'Train: {tr_loss:.4f} | {tr_acc:.4f}  ||  Val: {val_loss:.4f} | {val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_efficientnet_traffic.pth')
        print('Saved best model')

print(f'\nBest Val Acc: {best_acc:.4f}')

## 7. Download Model

In [ ]:
from google.colab import files
files.download('best_efficientnet_traffic.pth')